In [ ]:
!pip install -U albumentations ultralytics fiftyone torchview torchinfo torchmetrics

# AI/ML Video Analysis Capstone: Advanced Comparative Analysis
This notebook is a compiled version of the VS Code project for Kaggle. It covers data augmentation, architectural introspection, feature analysis, fine-tuning, and final benchmark evaluations.

## 1. Global Imports & Setup
Setting up the environment, device detection, and core libraries.

In [ ]:
import os
import glob
import numpy as np
import cv2
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.image as mpimg
import urllib.request
import zipfile
import tarfile
import time
import xml.etree.ElementTree as ET
from abc import ABC, abstractmethod
from torchvision.datasets import VOCDetection
from torchvision.transforms import functional as F
from torch.utils.data import Dataset, ConcatDataset, DataLoader
from albumentations.pytorch import ToTensorV2
import albumentations as A
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from ultralytics import YOLO
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights,
    ssd300_vgg16, SSD300_VGG16_Weights,
    retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights,
    fcos_resnet50_fpn, FCOS_ResNet50_FPN_Weights
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Part 2: Utility Modules
Core functional units for metrics, video processing, and model analysis.

### 2.1 Performance Metrics
Functions for plotting FPS comparisons.

In [ ]:
def plot_fps_comparison(results_dict, save_path="output/fps_comparison.png"):
    models = list(results_dict.keys())
    fps_vals = list(results_dict.values())
    plt.figure(figsize=(8, 5))
    plt.bar(models, fps_vals, color=['blue', 'orange', 'green'][:len(models)])
    plt.ylabel('Frames Per Second (FPS)')
    plt.title('Inference Speed Comparison')
    plt.savefig(save_path)
    plt.close()
    print(f"FPS comparison plot saved to {save_path}")

### 2.2 Video Streaming Utils
Logic for frame-by-frame inference on video files and webcam streams.

In [ ]:
def process_video_stream(model, video_path, out_path, num_frames=30):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): raise ValueError(f"Error opening video: {video_path}")
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))
    start_time = time.time(); frame_count = 0
    while cap.isOpened() and frame_count < num_frames:
        ret, frame = cap.read()
        if not ret: break
        annotated_frame = model.predict_and_annotate(frame)
        out.write(annotated_frame); frame_count += 1
    end_time = time.time(); cap.release(); out.release()
    return frame_count / (end_time - start_time) if frame_count > 0 else 0

### 2.3 VOC Dataset Evaluation
mAP calculation using torchmetrics on Pascal VOC.

In [ ]:
VOC_CLASSES = ["background", "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat", "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor"]
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(VOC_CLASSES)}

def parse_voc_xml(node):
    res = []
    objects = node.get("object", [])
    if not isinstance(objects, list): objects = [objects]
    for obj in objects:
        bndbox = obj.get("bndbox")
        if not bndbox: continue
        res.append({"name": obj.get("name"), "bbox": [int(bndbox.get("xmin")), int(bndbox.get("ymin")), int(bndbox.get("xmax")), int(bndbox.get("ymax"))]})
    return res

def evaluate_model_on_voc(model_wrapper, dataset_root='./data/voc', num_samples=50):
    dataset = VOCDetection(root=dataset_root, year='2012', image_set='val', download=True)
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    device, model = model_wrapper.device, model_wrapper.model
    model.eval()
    for i in range(min(num_samples, len(dataset))):
        img, target_dict = dataset[i]
        objects = parse_voc_xml(target_dict['annotation'])
        gt_boxes, gt_labels = [], []
        for obj in objects:
            if obj['name'] in CLASS_TO_IDX:
                gt_boxes.append(obj['bbox']); gt_labels.append(CLASS_TO_IDX[obj['name']])
        if not gt_boxes: continue
        target = [dict(boxes=torch.tensor(gt_boxes, dtype=torch.float32).to(device), labels=torch.tensor(gt_labels, dtype=torch.int64).to(device))]
        img_tensor = F.to_tensor(img).to(device)
        with torch.no_grad():
            if hasattr(model, 'predict'):
                results = model(img, verbose=False)
                preds = [dict(boxes=results[0].boxes.xyxy.to(device), scores=results[0].boxes.conf.to(device), labels=results[0].boxes.cls.to(torch.int64).to(device))]
            else:
                preds = model([img_tensor])
                keep = preds[0]['scores'] > 0.1
                preds = [dict(boxes=preds[0]['boxes'][keep], scores=preds[0]['scores'][keep], labels=preds[0]['labels'][keep])]
        metric.update(preds, target)
    results = metric.compute()
    return {'mAP@50': results['map_50'].item(), 'mAP@50-95': results['map'].item()}

### 2.4 Feature Analysis (t-SNE/PCA)
Visualizing high-dimensional latent representations.

In [ ]:
def perform_tsne_pca_analysis(features, labels=None, output_dir="output/feature_analysis"):
    os.makedirs(output_dir, exist_ok=True)
    pca = PCA(n_components=2)
    pca_res = pca.fit_transform(features)
    plt.figure(figsize=(8,6)); plt.scatter(pca_res[:,0], pca_res[:,1], c=labels, cmap='tab10', alpha=0.7); plt.title('PCA Analysis'); plt.show()
    tsne = TSNE(n_components=2, perplexity=min(30, max(5, len(features)//3)), random_state=42)
    tsne_res = tsne.fit_transform(features)
    plt.figure(figsize=(8,6)); plt.scatter(tsne_res[:,0], tsne_res[:,1], c=labels, cmap='tab10', alpha=0.7); plt.title('t-SNE Analysis'); plt.show()

### 2.5 Fine-Tuning Engine
Standard training loops for domain adaptation.

In [ ]:
def train_one_epoch(model_wrapper, optimizer, data_loader, device, epoch):
    model = model_wrapper.model; model.train(); total_loss = 0
    for images, targets in data_loader:
        images = list(img.to(device) for img in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        optimizer.zero_grad(); losses.backward(); optimizer.step()
        total_loss += losses.item()
    return total_loss / len(data_loader)

def finetune_model(model_wrapper, dataset, num_epochs=2, batch_size=2):
    device, model = model_wrapper.device, model_wrapper.model
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
    for epoch in range(num_epochs):
        loss = train_one_epoch(model_wrapper, optimizer, loader, device, epoch)
        print(f"Epoch {epoch} loss: {loss:.4f}")

### 2.6 Architecture Visualization
Extracting CNN feature maps and plotting network graphs.

In [ ]:
class LayerVisualizer:
    def __init__(self, model, output_dir='output/layer_visuals'):
        self.model = model; self.output_dir = output_dir; os.makedirs(output_dir, exist_ok=True)
        self.activations = {}; self.hooks = []
    def hook_fn(self, name):
        def fn(m, i, o): self.activations[name] = o.detach()
        return fn
    def register_hooks(self, layer_names):
        for name, m in self.model.named_modules():
            if name in layer_names: self.hooks.append(m.register_forward_hook(self.hook_fn(name)))
    def remove_hooks(self):
        for h in self.hooks: h.remove()
        self.hooks = []
    def plot_architecture(self, filename='arch', input_size=(1, 3, 640, 640)):
        try:
            from torchview import draw_graph
            graph = draw_graph(self.model, input_size=input_size, expand_nested=True, depth=5)
            graph.visual_graph.format = 'png'
            graph.visual_graph.render(os.path.join(self.output_dir, filename), cleanup=True)
            return os.path.join(self.output_dir, filename + '.png')
        except: return None

# Part 3: Data Engineering
Handling augmentations and multi-source dataset integration.

### 3.1 Augmentation Pipelines
Albumentations config for training and validation.

In [ ]:
COCO_CLASSES = ['person','bicycle','car','motorcycle','airplane','bus','train','truck','boat','traffic light','fire hydrant','stop sign','parking meter','bench','bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe','backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard','sports ball','kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza','donut','cake','chair','couch','potted plant','bed','dining table','toilet','tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven','toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear','hair drier','toothbrush']
VOC_CLASSES = ['background','aeroplane','bicycle','bird','boat','bottle','bus','car','cat','chair','cow','diningtable','dog','horse','motorbike','person','pottedplant','sheep','sofa','train','tvmonitor']

def get_train_transforms():
    return A.Compose([A.LongestMaxSize(max_size=640), A.PadIfNeeded(640, 640, border_mode=cv2.BORDER_CONSTANT, value=114), A.HorizontalFlip(p=0.5), A.RandomBrightnessContrast(p=0.5), A.Normalize(), ToTensorV2()], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

### 3.2 Dataset Download & Loader
Automated downloading of standard benchmarks.

In [ ]:
class DatasetLoader:
    def __init__(self, data_dir="data"):
        self.data_dir = data_dir; os.makedirs(data_dir, exist_ok=True)
    def load_voc(self, year='2012'):
        return VOCDetection(root=os.path.join(self.data_dir, 'voc'), year=year, image_set='val', download=True)
    def load_coco8(self):
        url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip"
        path = os.path.join(self.data_dir, "coco8.zip"); extract = os.path.join(self.data_dir, "coco8")
        if not os.path.exists(extract):
            urllib.request.urlretrieve(url, path)
            with zipfile.ZipFile(path, 'r') as z: z.extractall(self.data_dir)
        return extract

# Part 4: Model Architectures
Defining the comparative suite of object detectors.

### 4.1 Base Model & YOLO
Abstract interfaces and Ultralytics YOLO wrappers.

In [ ]:
class BaseModel(ABC):
    def __init__(self, device): self.device = device; self.model = None
    @abstractmethod
    def load(self): pass
    @abstractmethod
    def predict_and_annotate(self, frame): pass

class YOLOModel(BaseModel):
    def __init__(self, device, weights='yolov8n.pt'): super().__init__(device); self.weights = weights
    def load(self): self.model = YOLO(self.weights).to(self.device)
    def predict_and_annotate(self, frame):
        res = self.model(frame, verbose=False, device=self.device)
        return res[0].plot()

### 4.2 Faster R-CNN, SSD, RetinaNet
Modified Torchvision architectures.

In [ ]:
class CustomRCNNModel(BaseModel):
    def load(self):
        self.model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT).to(self.device); self.model.eval()
    def predict_and_annotate(self, frame):
        t = F.to_tensor(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).to(self.device)
        with torch.no_grad(): pred = self.model([t])[0]
        out = frame.copy()
        for i, b in enumerate(pred['boxes']):
            if pred['scores'][i] > 0.5:
                bx = b.cpu().numpy().astype(int); cv2.rectangle(out, (bx[0], bx[1]), (bx[2], bx[3]), (0, 255, 0), 2)
        return out

class SSDModel(BaseModel):
    def load(self): self.model = ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT).to(self.device); self.model.eval()
    def predict_and_annotate(self, frame):
        t = F.to_tensor(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).to(self.device)
        with torch.no_grad(): pred = self.model([t])[0]
        out = frame.copy()
        for i, b in enumerate(pred['boxes']):
            if pred['scores'][i] > 0.5:
                bx = b.cpu().numpy().astype(int); cv2.rectangle(out, (bx[0], bx[1]), (bx[2], bx[3]), (0, 255, 0), 2)
        return out

# Part 5: Pipeline Execution
Running the full end-to-end evaluation.

### 5.1 Data Initialization

In [ ]:
loader = DatasetLoader()
voc2012 = loader.load_voc()
coco8_dir = loader.load_coco8()
print(f"Data ready. VOC images: {len(voc2012)}")

### 5.2 Model Benchmarking
Loading models and running accuracy/speed tests.

In [ ]:
models = {
    'YOLOv8n': YOLOModel(device, 'yolov8n.pt'),
    'Faster R-CNN': CustomRCNNModel(device),
    'SSD300': SSDModel(device)
}
for name, m in models.items(): m.load()

acc_results = {}
for name, m in models.items():
    acc_results[name] = evaluate_model_on_voc(m, num_samples=10)

df = pd.DataFrame(acc_results).T
display(df)